# Going Modular

**concept**: turn useful notebook code cells into reuseable python files.
we will need two notebooks:
1. cell mode: run as a traditional jupyter notebook/ google colab.
2. script mode: same as cell mode but with added functionality to turn useful code into python scripts, ex: train.py data_setup.py model_builder.py

cell mode: regular notebook. cells contain text or code.

difference b/w cell and script mode? cell mode is a more cleaned version of regular notebook of most useful code running one cell at a time.
script mode has extra subsection (e.g. 2.1 3.1...) for turning cell code into scipt code.

**docstring:** """one line summary of program end with period ->
one line blank ->
rest overall description of program ->
 brief description of classes/fn/usage examples. (args, returns, raises(exception))"""

In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Get data

In [7]:
#!rm -rf models/

In [8]:
import os
import zipfile

from pathlib import Path
import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If image folder doesn't exist, download it and prepare it
if image_path.is_dir():
  print(f"{image_path} directory exists.")
else:
  print(f"Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

  # Download data
  with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip")
    print("Downloading data...")
    f.write(request.content)

  # Unzip data
  with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping data....")
    zip_ref.extractall(image_path)

  # Remove zip file
  os.remove(data_path / "pizza_steak_sushi.zip")

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping data....


In [9]:
# setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

## 2. Create Datasets and Dataloaders

In [10]:
from torchvision import datasets, transforms

# Create simple transforms
data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Use image folder to create datasets
train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform,
                                  target_transform=None)

test_data = datasets.ImageFolder(root=test_dir,
                                  transform=data_transform)

print(f"Train data: \n{train_data}\nTest data: \n{test_data}")

Train data: 
Dataset ImageFolder
    Number of datapoints: 225
    Root location: data/pizza_steak_sushi/train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Test data: 
Dataset ImageFolder
    Number of datapoints: 75
    Root location: data/pizza_steak_sushi/test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )


In [11]:
#Turn Train and Test Datasets into Dataloaders
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=1,
                              num_workers=1,
                              shuffle=True)

test_dataloader =  DataLoader(dataset=test_data,
                              batch_size=1,
                              num_workers=1,
                              shuffle=True)

train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x7a97964e0750>,
 <torch.utils.data.dataloader.DataLoader at 0x7a989cf2e910>)

In [12]:
# Check out a single image size/shape
img, label = next(iter(train_dataloader))
print(f"Image shape: {img.shape} -> [batch, color, h, w]")
print(f"label shape: {label.shape}")

Image shape: torch.Size([1, 3, 64, 64]) -> [batch, color, h, w]
label shape: torch.Size([1])


###2.1 Create Datasets and DataLoaders (Script mode)  
use Jupyter magic fn to create `.py` file for creating DataLoaders.
How? we can save a code cell's contents to a file using file Jupyter magic `%%writefile filename -`

In [13]:
# create directory for going modular script
!rm -rf going_modular/
os.makedirs("going_modular")

In [14]:
%%writefile going_modular/data_setup.py
"""
Contains functionality for creating PyTorch DataLoader's for image classification data.
"""
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

num_workers = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int = num_workers,
):
  """ Create training and testing DataLoaders.

  Takes train and test directory paths and turn them into pytorch datasets
  and then pytorch dataloaders.

  Args:
    train_dir: path to train dir.
    test_dir: path to test dir.
    transform: torchvision transforms to perform on train and test data.
    batch_size: no. of samples per patch in each of dataloaders.
    num_workers: an integer for no. of workers per dataloader.

    Returns:
      A tuple of (train_dataloader, test_dataloader, class_names).
      where class_names is a list of target classes.

    Example usage:
      train_dataloader, test_dataloader, class_names = create_dataloaders(train_dir=path/to/train_dir,
      test_dir=path/to/test_dir,
      transform=some_transform,
      batch_size=32,
      num_workers=1)

  """
  # Use ImageFolder to create datasets
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get the class_names
  class_names = train_data.classes

  # Turn images into DataLoaders
  train_dataloader = DataLoader(train_data,
                                batch_size=batch_size,
                                num_workers=num_workers,
                                shuffle=True,
                                pin_memory=True)
  test_dataloader = DataLoader(test_data,
                               batch_size=batch_size,
                               num_workers=num_workers,
                               shuffle=True,
                               pin_memory=False)

  return train_dataloader, test_dataloader, class_names

Writing going_modular/data_setup.py


In [15]:
from going_modular import data_setup
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=data_transform,
                                                                               batch_size=32,
                                                                               )

train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7a97952df8d0>,
 ['pizza', 'steak', 'sushi'])

## 3. Making a model (TinyVGG)

In [16]:
import torch
from torch import nn

class TinyVGG(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels= input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*13*13,
                  out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [17]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate an instance of the model
torch.manual_seed(42)
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(train_data.classes)).to(device)

model_0

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1690, out_features=3, bias=True)
  )
)

In [18]:
# To test the model we do a single/dummy forward pass to check if model works
img_batch, label_batch = next(iter(train_dataloader))

# 1. Get a batch of images and labels from the dataloader
img_0, label_0 = img_batch[0].unsqueeze(dim=0), label_batch[0]
print(f"Single image shape: {img_0.shape}\n")

# 2. Perform a forward pass on a single image
model_0.eval()

with torch.inference_mode():
  pred = model_0(img_0.to(device))

# 4. Print out what is happening and convert model logits -> pred probs -> labels
print(f"the pred logit: {pred}\n")
print(f"the pred prob: {torch.softmax(pred, dim=1)}\n")
print(f"the pred label: {torch.argmax(torch.softmax(pred, dim=1), dim=1)} \n")
print(f"Actual labe: {label_0}")

Single image shape: torch.Size([1, 3, 64, 64])

the pred logit: tensor([[ 0.0208, -0.0020,  0.0095]], device='cuda:0')

the pred prob: tensor([[0.3371, 0.3295, 0.3333]], device='cuda:0')

the pred label: tensor([0], device='cuda:0') 

Actual labe: 0


###3.1 Making a model (TinyVGG) with a script (`model_builder.py`)
so we can import

>https://horace.io/brrr_intro.html

> https://poloclub.github.io/cnn-explainer/

In [19]:
%%writefile going_modular/model_builder.py
"""
Contains PyTorch model code to instantiate a TinyVGG model from the CNN explainer website.
"""
import torch
from torch import nn

class TinyVGG(nn.Module):
  """
  Create the TinyVGG architecture.

  Replicate the TinyVGG model from the CNN explainer website in PyTorch.

  Args:
    input_shape: an integer indicating number of input channels.
    hidden_units: an integer indicating number of hidden units between layers.
    output_shape: an integer insicating number of output channels.
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels= input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*13*13,
                  out_features=output_shape)
    )

  def forward(self, x: torch.Tensor):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

Writing going_modular/model_builder.py


In [20]:
import torch

from going_modular import model_builder
device = "cuda" if torch.cuda.is_available else "cpu"

# Instantiate a model from the model_builder
torch.manual_seed(42)
model_1 = model_builder.TinyVGG(input_shape=3,
                                hidden_units=10,
                                output_shape=len(class_names)).to(device)

# Dummy forward pass
model_1.eval()

with torch.inference_mode():
  y_logit = model_1(img_0.to(device))

y_prob = torch.softmax(y_logit, dim=1)
y_label = torch.argmax(y_prob, dim=1)

print(
    f"pred logit: \n{y_logit} \n \n"
    f"pred prob: \n{y_prob} \n \n"
    f"pred label: \n{y_label} \n \n"
    f"Actual label: \n{label_0}"
)

pred logit: 
tensor([[ 0.0208, -0.0020,  0.0095]], device='cuda:0') 
 
pred prob: 
tensor([[0.3371, 0.3295, 0.3333]], device='cuda:0') 
 
pred label: 
tensor([0], device='cuda:0') 
 
Actual label: 
0


In [21]:
#%%writefile going_modular/model_builder.py

print("hello")

hello


In [22]:
#!python going_modular/model_builder.py

##4. Creating `train_step()` and `test_step` functions and combine them with `train()` function

`train_step()`

In [23]:
from typing import Tuple

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]: # it returns a tuple of loss and accuracy
  """Trains a PyTorch model for a single epoch.

  Turns a target PyTorch model to training mode and then runs through all
  required training steps (forward pass, loss calculation, optimizer step.)
'
  Args:
    model: A PyTorch model to be trained on.
    dataloader: A dataloader instance for the model to be trained on.
    loss_fn: A Pytorch loss function to minimize.
    optimizer: A pytorch optimizer to help loss function to minimize.
    device: A target device to compute on (e.g. "Cuda" or "cpu")

  Returns:
    A tuple of training loss and training accuracy metrices, in
    the form (train_loss, train_acc) e.x.(0.3335, 0.8456)

  Example usage:
    train_loss, train_acc = train_step(model=model_name,
               dataloader=path/to/dataloader,
               loss_fn=loss-fn,
               optimizer=Optimizer,
               device=device)
  """
  # Put model in train mode
  model.train()

  # Setup model loss and acc values
  train_loss, train_acc = 0, 0

  #loop through dataloader batchs
  for batch, (X, y) in enumerate(dataloader):
    # Send data to target device
    X, y = X.to(device), y.to(device)

    # 1. Forward pass
    y_pred = model(X)

    # 2. Calculate and accumulate the loss
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    # Calculate and accumulate accuracy metric across all branches
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class == y).sum().item() /len(y_pred)

  # Adjuct metrics to get average loss and accuracy per batch
  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc


`test_step()`

In [24]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to eval mode and then perform
    a forward pass on testing dataset and calculate loss and accuracy.
    '
    Args:
        model: A PyTorch model to be tested on.
        dataloader: A dataloader instance for the model to be tested on.
        loss_fn: A Pytorch loss function to calculate loss on the test data.
        device: A target device to compute on (e.g. "Cuda" or "cpu")

    Returns:
        A tuple of testing loss and testing accuracy metrices, in
        the form (test_loss, test_acc) e.x.(0.3335, 0.8456)

    Example usage:
        test_loss, test_acc = test_step(model=model_name,
                                        dataloader=path/to/dataloader,
                                        loss_fn=loss-fn,
                                        device=device)
    """

    # Put model in eval mode
    model.eval()

    # Setup test loss and accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logit = model(X)

            # 2. Calculate and accumulate the loss
            loss = loss_fn(test_pred_logit, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_label = test_pred_logit.argmax(dim=1)
            test_acc += ((test_pred_label == y).sum().item() / len(test_pred_label))

        # Adjuct metrics to get average loss and accuracy per batch
        test_loss += test_loss / len(dataloader)
        test_acc += test_acc / len(dataloader)

    return test_loss, test_acc

`train()`

In [25]:
from typing import Dict, List
from tqdm.auto import tqdm

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device) -> Dict[str, List[float]]:
    """Runs training and testing for a number of epochs and tracks metrics.

    Args:
        model: A PyTorch model to be tested.
        train_dataloader: A train dataloader instance for the train dataset.
        test_dataloader: A test dataloader instance for the test dataset.
        loss_fn: A PyTorch loss function to minimiz.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        epochs: An integer indicating the number of epochs to be trained for.
        device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A Dictionary of training and testing loss as well as training and testing accuracy metrics,
        each metric has values in the form of list, in the form:
            {
              "train_loss": [...],
              "train_acc": [...],
              "test_loss": [...],
              "test_acc": [...]
            }

        Example, if number of epochs is 3:
            {
              "train_loss": [0.4352, 0.3456, 0.2357],
              "train_acc": [0.8765, 0.8854, 0.8976],
              "test_loss": [0.7658, 0.6789, 0.5498],
              "test_acc": [0.7895, 0.7890, 0.7998]
            }

    Example usage:
        Results = train(
            model=model,
            train_dataloader= path/to/train/DataLoader,
            test_dataloader= path/to/test/DataLoader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs= 5,
            device=device
        )
    """

    # Empty result dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []}

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)

        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        # Print out what is happening
        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f} | "
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results

### 4.1 Turn training functions into a script (`engine.py`)

> remember always to import all necessary libraries into the script or it will throw error

In [43]:
%%writefile going_modular/engine.py
"""
Contains functions for training and testing PyTorch.
"""

import torch
from typing import Tuple, List, Dict
from tqdm.auto import tqdm

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then runs through all
    required training steps (forward pass, loss calculation, optimizer step.)

    Args:
        model: A PyTorch model to be trained on.
        dataloader: A dataloader instance for the model to be trained on.
        loss_fn: A PyTorch loss function to minimize.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        device: A target device to compute on (e.g. "cuda" or "cpu")

    Returns:
        A tuple of training loss and training accuracy metrics, in
        the form (train_loss, train_acc) e.g. (0.3335, 0.8456)

    Example usage:
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
    """
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward pass
        y_pred = model(X)

        # Calculate and accumulate the loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    train_loss /= len(dataloader)
    train_acc /= len(dataloader)

    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to eval mode and then performs
    a forward pass on the testing dataset and calculates loss and accuracy.

    Args:
        model: A PyTorch model to be tested.
        dataloader: A dataloader instance for the test dataset.
        loss_fn: A PyTorch loss function.
        device: A target device to compute on (e.g. "cuda" or "cpu")

    Returns:
        A tuple of testing loss and testing accuracy metrics, in
        the form (test_loss, test_acc) e.g. (0.3335, 0.8456)

    Example usage:
        test_loss, test_acc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )
    """
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            # Forward pass
            test_pred_logits = model(X)

            # Loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Accuracy
            test_pred_labels = torch.argmax(test_pred_logits, dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(test_pred_labels)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

    return test_loss, test_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device) -> Dict[str, List[float]]:
    """Runs training and testing for a number of epochs and tracks metrics.

    Args:
        model: A PyTorch model to be tested.
        train_dataloader: A train dataloader instance for the train dataset.
        test_dataloader: A test dataloader instance for the test dataset.
        loss_fn: A PyTorch loss function to minimiz.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        epochs: An integer indicating the number of epochs to be trained for.
        device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A Dictionary of training and testing loss as well as training and testing accuracy metrics,
        each metric has values in the form of list, in the form:
            {
              "train_loss": [...],
              "train_acc": [...],
              "test_loss": [...],
              "test_acc": [...]
            }

        Example, if number of epochs is 3:
            {
              "train_loss": [0.4352, 0.3456, 0.2357],
              "train_acc": [0.8765, 0.8854, 0.8976],
              "test_loss": [0.7658, 0.6789, 0.5498],
              "test_acc": [0.7895, 0.7890, 0.7998]
            }


    Example usage:
        Results = train(
            model=model,
            train_dataloader= path/to/train/DataLoader,
            test_dataloader= path/to/test/DataLoader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs= 5,
            device=device
        )
    """

    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        test_loss, test_acc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results

Overwriting going_modular/engine.py


In [44]:
from going_modular import engine

#Results = engine.train(model=model_0, train_dataloader=train_dataloader, test_dataloader=test_dataloader, loss_fn=loss_fn, optimizer=optimizer, epochs=NUM_EPOCHS, device=device)
#Results

## 5. Creating a function to save the model

In [45]:
from pathlib import Path
import torch  # Make sure to import torch

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
    """
    A function to save PyTorch model to target directory.

    Args:
        model: A target PyTorch model to save.
        target_dir: A string indicating the target directory to save model to.
        model_name: A string indicating the filename for the model; should include 'pth' or 'pt'.

    Example Usage:
        save_model(model=model,
                   target_dir="models",
                   model_name="name.pth")
    """

    # Create target directory
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    # Create model save path
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), \
        "model_name should end with `.pth` or `.pt`"
    model_save_path = target_dir_path / model_name

    # Save the model state_dict()
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(), f=model_save_path)


### 5.1 Turn the utility function into script (`utils.py`)
utils in python is generally reserved for various utility functions
here, we have one utility function `save_model()`, but there are many more...


In [46]:
%%writefile going_modular/utils.py
"""File Contains various utility functions for PyTorch to train the model.
"""
from pathlib import Path
import torch
def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
    """
    A function to save PyTorch model to target directory.

    Args:
        model: A target PyTorch model to save.
        target_dir: A string indicating the target directory to save model to.
        model_name: A string indicating the filename for the model; should include 'pth' or 'pt'.

    Example Usage:
        save_model(model=model,
                   target_dir="models",
                   model_name="name.pth")
    """

    # Create target directory
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    # Create model save path
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), \
        "model_name should end with `.pth` or `.pt`"
    model_save_path = target_dir_path / model_name

    # Save the model state_dict()
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(), f=model_save_path)

Overwriting going_modular/utils.py


##6. Train, evaluate, and save the model

In [47]:
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 5

# Recreate an instance of TinyVGG
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(train_data.classes)).to(device)

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                            lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        loss_fn=loss_fn,
                        optimizer=optimizer,
                        epochs=NUM_EPOCHS,
                        device=device)

# End the timer  and print how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time} second.")

# Save the model
save_model(model=model_0,
           target_dir="models",
           model_name="model_0.pth")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.1063 | train_acc: 0.3047 | test_loss: 4.3916 | test_acc: 1.3636 | 
Epoch: 2 | train_loss: 1.0835 | train_acc: 0.4453 | test_loss: 4.3112 | test_acc: 1.6894 | 
Epoch: 3 | train_loss: 1.1063 | train_acc: 0.2812 | test_loss: 4.3272 | test_acc: 1.5303 | 
Epoch: 4 | train_loss: 1.0755 | train_acc: 0.4766 | test_loss: 4.3107 | test_acc: 2.0189 | 
Epoch: 5 | train_loss: 1.0664 | train_acc: 0.4062 | test_loss: 4.2074 | test_acc: 2.0152 | 
[INFO] Total training time: 4.661920312000007 second.
[INFO] Saving model to: models/model_0.pth


 This code is running one cell at once, now we will make it in a script mode.

### 6.1 Train, evaluate, and save the model (script mode) -> `train.py`
`train.py` leverage all of other code code scipts to train a PyTorch model.

to replicate functionality of notebook 04 in one line.

In [52]:
%%writefile going_modular/train.py
"""
Trains a PyTorch classification model using device-agnostic code.
"""

import os
import torch
from timeit import default_timer as timer
from torchvision import transforms
import data_setup, engine, model_builder, utils
from tqdm.auto import tqdm

#import argparse

# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# Setup directories
train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create transforms
data_transform = transforms.Compose([
                                    transforms.Resize((64, 64)),
                                    transforms.ToTensor()
                                    ])

# Create Dataloader's and get class_names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# create an instance of TinyVGG
model = model_builder.TinyVGG(input_shape=3,
                              hidden_units=HIDDEN_UNITS,
                              output_shape=len(class_names)).to(device)

# Setup loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(),
                            lr=0.001)

# Start the timer
start_time = timer()

# Start training with help from engine.py
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

# End the timer  and print how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time} second.")

# Save the model to file
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular.pth")

Overwriting going_modular/train.py


In [53]:
 !python going_modular/train.py

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.1063 | train_acc: 0.3047 | test_loss: 1.0979 | test_acc: 0.3409
 20% 1/5 [00:01<00:04,  1.10s/it]Epoch: 2 | train_loss: 1.0835 | train_acc: 0.4453 | test_loss: 1.0778 | test_acc: 0.4223
 40% 2/5 [00:01<00:02,  1.05it/s]Epoch: 3 | train_loss: 1.1063 | train_acc: 0.2812 | test_loss: 1.0818 | test_acc: 0.3826
 60% 3/5 [00:02<00:01,  1.11it/s]Epoch: 4 | train_loss: 1.0755 | train_acc: 0.4766 | test_loss: 1.0779 | test_acc: 0.5047
 80% 4/5 [00:03<00:00,  1.09it/s]Epoch: 5 | train_loss: 1.0665 | train_acc: 0.4023 | test_loss: 1.0519 | test_acc: 0.5038
100% 5/5 [00:04<00:00,  1.09it/s]
[INFO] Total training time: 4.583693140000037 second.
[INFO] Saving model to: models/05_going_modular.pth
